In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
from pyspark.sql import functions as F
df_sessions_raw = spark.table("lh_bronze_game.sessions_raw")

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 3, Finished, Available, Finished, False)

In [2]:
print("Row count:", df_sessions_raw.count())
print("Column count:", len(df_sessions_raw.columns))
df_sessions_raw.printSchema()
display(df_sessions_raw.limit(5))

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 4, Finished, Available, Finished, False)

Row count: 402936
Column count: 7
root
 |-- days_since_install: long (nullable = true)
 |-- player_id: string (nullable = true)
 |-- session_duration_minutes: double (nullable = true)
 |-- session_end: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- session_number: long (nullable = true)
 |-- session_start: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 51f20047-3b03-4ae7-84a6-28d4f4f643c3)

In [3]:
df_sessions_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_sessions_raw.columns
]).show(truncate=False)


StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 5, Finished, Available, Finished, False)

+------------------+---------+------------------------+-----------+----------+--------------+-------------+
|days_since_install|player_id|session_duration_minutes|session_end|session_id|session_number|session_start|
+------------------+---------+------------------------+-----------+----------+--------------+-------------+
|0                 |0        |0                       |0          |418       |0             |0            |
+------------------+---------+------------------------+-----------+----------+--------------+-------------+



In [4]:
total_sessions = df_sessions_raw.count()

unique_session_ids = (
    df_sessions_raw
    .filter(F.col("session_id").isNotNull())
    .select("session_id")
    .distinct()
    .count()
)

non_null_session_count = (
    df_sessions_raw
    .filter(F.col("session_id").isNotNull())
    .count()
)

duplicate_session_ids = non_null_session_count - unique_session_ids

print("Total sessions:", total_sessions)
print("Non-null session_id:", non_null_session_count)
print("Unique session_id:", unique_session_ids)
print("Duplicate session_id:", duplicate_session_ids)

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 6, Finished, Available, Finished, False)

Total sessions: 402936
Non-null session_id: 402518
Unique session_id: 402127
Duplicate session_id: 391


In [5]:
duplicate_ids = (
    df_sessions_raw
    .filter(F.col("session_id").isNotNull())
    .groupBy("session_id")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_ids.limit(10))

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e24839d4-d418-499a-971c-9e9022e9b567)

In [6]:
duplicate_rows = (
    df_sessions_raw
    .join(
        duplicate_ids.select("session_id"),
        on="session_id",
        how="inner"
    )
    .orderBy("session_id")
)

display(duplicate_rows.limit(20))

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6e18ea0c-fa08-45f1-814e-34df1a301fd9)

In [7]:
exact_duplicate_count = (
    df_sessions_raw.count()
    - df_sessions_raw.dropDuplicates().count()
)

print("Exact duplicate rows:", exact_duplicate_count)


StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 9, Finished, Available, Finished, False)

Exact duplicate rows: 391


In [8]:
df_sessions_clean = df_sessions_raw.dropDuplicates()

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 10, Finished, Available, Finished, False)

In [9]:
print("Raw row count:", df_sessions_raw.count())
print("Clean row count:", df_sessions_clean.count())

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 11, Finished, Available, Finished, False)

Raw row count: 402936
Clean row count: 402545


In [10]:
df_sessions_clean = (
    df_sessions_clean
    .withColumn(
        "session_id",
        F.when(
            F.col("session_id").isNull(),
            F.concat(
                F.lit("MISSING_"),
                F.monotonically_increasing_id()
            )
        ).otherwise(F.col("session_id"))
    )
)

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 12, Finished, Available, Finished, False)

In [11]:
df_sessions_clean.filter(
    F.col("session_id").isNull()
).count()

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 13, Finished, Available, Finished, False)

0

In [12]:
df_time_check = (
    df_sessions_clean
    .withColumn(
        "session_start_parsed",
        F.to_timestamp("session_start")
    )
    .withColumn(
        "session_end_parsed",
        F.to_timestamp("session_end")
    )
)

df_time_check.select(
    F.count(
        F.when(F.col("session_start_parsed").isNull(), 1)
    ).alias("invalid_session_start"),

    F.count(
        F.when(F.col("session_end_parsed").isNull(), 1)
    ).alias("invalid_session_end")
).show()

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 14, Finished, Available, Finished, False)

+---------------------+-------------------+
|invalid_session_start|invalid_session_end|
+---------------------+-------------------+
|                    0|                  0|
+---------------------+-------------------+



In [13]:
df_sessions_clean = (
    df_sessions_clean
    .withColumn(
        "session_start",
        F.to_timestamp("session_start")
    )
    .withColumn(
        "session_end",
        F.to_timestamp("session_end")
    )
)

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 15, Finished, Available, Finished, False)

In [14]:
df_sessions_clean.printSchema()

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 16, Finished, Available, Finished, False)

root
 |-- days_since_install: long (nullable = true)
 |-- player_id: string (nullable = true)
 |-- session_duration_minutes: double (nullable = true)
 |-- session_end: timestamp (nullable = true)
 |-- session_id: string (nullable = true)
 |-- session_number: long (nullable = true)
 |-- session_start: timestamp (nullable = true)



In [15]:
invalid_time_order = (
    df_sessions_clean
    .filter(F.col("session_end") < F.col("session_start"))
    .count()
)

print("session_end < session_start:", invalid_time_order)

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 17, Finished, Available, Finished, False)

session_end < session_start: 398


In [16]:
invalid_time_rate = invalid_time_order / df_sessions_clean.count() * 100

print("Invalid time rate (%):", invalid_time_rate)

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 18, Finished, Available, Finished, False)

Invalid time rate (%): 0.0988709336844328


In [17]:
df_sessions_clean = (
    df_sessions_clean
    .filter(F.col("session_end") >= F.col("session_start"))
)

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 19, Finished, Available, Finished, False)

In [18]:
print(
    df_sessions_clean
    .filter(F.col("session_end") < F.col("session_start"))
    .count()
)

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 20, Finished, Available, Finished, False)

0


In [19]:
df_sessions_clean.select(
    F.min("session_duration_minutes").alias("min_duration"),
    F.max("session_duration_minutes").alias("max_duration"),
    F.avg("session_duration_minutes").alias("avg_duration")
).show()

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 21, Finished, Available, Finished, False)

+------------+------------+-----------------+
|min_duration|max_duration|     avg_duration|
+------------+------------+-----------------+
|         0.5|     1497.44|7.804794987902434|
+------------+------------+-----------------+



In [20]:
df_sessions_clean = (
    df_sessions_clean
    .filter(F.col("session_duration_minutes") <= 1440)
)

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 22, Finished, Available, Finished, False)

In [21]:
df_sessions_clean.filter(
    F.col("session_duration_minutes") > 1440
).count()

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 23, Finished, Available, Finished, False)

0

In [22]:
df_sessions_clean.select(
    F.min("days_since_install").alias("min_days_since_install"),
    F.max("days_since_install").alias("max_days_since_install"),
    F.min("session_number").alias("min_session_number"),
    F.max("session_number").alias("max_session_number")
).show()

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 24, Finished, Available, Finished, False)

+----------------------+----------------------+------------------+------------------+
|min_days_since_install|max_days_since_install|min_session_number|max_session_number|
+----------------------+----------------------+------------------+------------------+
|                     0|                    89|                 1|               121|
+----------------------+----------------------+------------------+------------------+



In [23]:
orphan_sessions = (
    df_sessions_clean.alias("s")
    .join(
        spark.table("lh_silver_game.players_clean").alias("p"),
        F.col("s.player_id") == F.col("p.player_id"),
        "left_anti"
    )
)

print("Players tablosunda bulunmayan session sayısı:", orphan_sessions.count())

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 25, Finished, Available, Finished, False)

Players tablosunda bulunmayan session sayısı: 0


In [24]:
print("Final row count:", df_sessions_clean.count())

df_sessions_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in [
        "player_id",
        "session_id",
        "session_start",
        "session_end",
        "session_duration_minutes",
        "session_number",
        "days_since_install"
    ]
]).show()

df_sessions_clean.printSchema()

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 26, Finished, Available, Finished, False)

Final row count: 402120
+---------+----------+-------------+-----------+------------------------+--------------+------------------+
|player_id|session_id|session_start|session_end|session_duration_minutes|session_number|days_since_install|
+---------+----------+-------------+-----------+------------------------+--------------+------------------+
|        0|         0|            0|          0|                       0|             0|                 0|
+---------+----------+-------------+-----------+------------------------+--------------+------------------+

root
 |-- days_since_install: long (nullable = true)
 |-- player_id: string (nullable = true)
 |-- session_duration_minutes: double (nullable = true)
 |-- session_end: timestamp (nullable = true)
 |-- session_id: string (nullable = true)
 |-- session_number: long (nullable = true)
 |-- session_start: timestamp (nullable = true)



In [25]:
df_sessions_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lh_silver_game.sessions_clean")

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 27, Finished, Available, Finished, False)

In [26]:
df_check = spark.table("lh_silver_game.sessions_clean")

print("Saved row count:", df_check.count())
display(df_check.limit(5))

StatementMeta(, 91301a17-7dfb-4539-b214-92b3fa35814d, 28, Finished, Available, Finished, False)

Saved row count: 402120


SynapseWidget(Synapse.DataFrame, 29386a4f-8f4a-4700-ac09-e7afb40c90fb)